## Practice Lab 20: Convolutional Neural Networks
In this lab we will look at how Convolutional Neural Networks for classification and regression. \
Based on Chapter 14 from Aurelien Geron's book, Hands-on Machine Learning with Scikit-Learn Keras & Tensorflow.\
Original code examples from book in github [here](https://github.com/ageron/handson-ml2)

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/dtrad/geoml_course/blob/master/Practice20_CNN.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

In [ ]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

try:
    # %tensorflow_version only exists in Colab.
    %tensorflow_version 2.x
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"
if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. CNNs can be very slow without a GPU.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware accelerator.")

# Add for GPU BEFORE JSON
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession

config = ConfigProto()
config.gpu_options.allow_growth = True
session = InteractiveSession(config=config)
####################################

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)
tf.random.set_seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt

Added two functions for plotting

In [ ]:
def plot_image(image):
    plt.imshow(image, cmap="gray", interpolation="nearest")
    plt.axis("off")

def plot_color_image(image):
    plt.imshow(image, interpolation="nearest")
    plt.axis("off")

## Exercise 1: CNN dimensions
Load the two images below and apply a vertical and a horizontal filter.\
Check all dimensions. Then plot one row of one of the figures before and after convolutions. \
Show how the convolution with the horizontal filter produces a smoothing along the rows.

In [ ]:
import numpy as np
from sklearn.datasets import load_sample_image
# Load sample images
china = load_sample_image("china.jpg") / 255
flower = load_sample_image("flower.jpg") / 255
images = np.array([china, flower])
print("batch_size, height, width, channels",images.shape)
batch_size, height, width, channels = images.shape
# Create 2 filters
filters = np.zeros(shape=(7, 7, channels, 2), dtype=np.float32)
filters[:, 3, :, 0] = 1  # vertical line
filters[3, :, :, 1] = 1  # horizontal line

In [ ]:
plot_image(china)

In [ ]:
print(china.shape)

In [ ]:
print(flower.shape)

In [ ]:
# show a row and a column of the original picture
plt.subplot(121);plt.plot(china[50,:,0]);plt.title('row')
plt.subplot(122);plt.plot(china[:,50,0]);plt.title('column')

In [ ]:
print(images.shape)

In [ ]:
print(filters.shape)

In [ ]:
plt.subplot(121);plot_image(filters[:,:,:,0])
plt.subplot(122);plot_image(filters[:,:,:,1])

In [ ]:
outputs = tf.nn.conv2d(images, filters, strides=1, padding="SAME")

In [ ]:
print(outputs.shape)

In [ ]:
print(images.shape)

In [ ]:
print(filters.shape)

In [ ]:
plot_color_image(outputs[0,:,:,1])

In [ ]:
# plot a row and a column before and after the vertical filter
plt.subplot(221);plt.plot(china[50,:,0]);plt.title('row');plt.ylabel('before')
plt.subplot(222);plt.plot(china[:,50,0]);plt.title('column')
plt.subplot(223);plt.plot(outputs[0,50,:,0]);plt.ylabel('after vert')
plt.subplot(224);plt.plot(outputs[0,:,50,0]);

In [ ]:
# plot a row and a column after the horizontal filter

plt.subplot(221);plt.plot(china[50,:,0]);plt.title('row');plt.ylabel('before')
plt.subplot(222);plt.plot(china[:,50,0]);plt.title('column')
plt.subplot(223);plt.plot(outputs[0,50,:,1]);plt.ylabel('after hz')
plt.subplot(224);plt.plot(outputs[0,:,50,1])

In [ ]:
plt.figure(figsize=(20,20))
plt.subplot(141),plt.imshow(china)
for i in range(3):
    print(i)
    plt.subplot(142+i);plt.imshow(china[:,:,i]);


In [ ]:
outputs = tf.nn.conv2d(images, filters, strides=1, padding="SAME")
plt.figure(figsize=(20,20))
plt.subplot(141),plt.imshow(china)
for i in range(2):
    print(i)
    plt.subplot(142+i);plt.imshow(outputs[0,:,:,i]);


In [ ]:
max_pool = keras.layers.MaxPool2D(pool_size=8)
outputspool=max_pool(outputs)
print(type(max_pool))
print(outputs.shape)
print(outputspool.shape)
plt.figure(figsize=(10,10))
plt.subplot(121);plt.imshow(outputs[0,:,:,0])
plt.subplot(122);plt.imshow(outputspool[0,:,:,0])

## Exercise 2: MNIST Classification with CNN vs 1D
Write conv2d for the mnist data set and compare with the 1D neural network we did in a previous lab.\
To treat each digit as an image, you need to add the feature axis, 1 in this case because it is black-white image.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train, X_valid = X_train_full[:-5000], X_train_full[-5000:]
y_train, y_valid = y_train_full[:-5000], y_train_full[-5000:]

X_mean = X_train.mean(axis=0, keepdims=True)
X_std = X_train.std(axis=0, keepdims=True) + 1e-7
X_train = (X_train - X_mean) / X_std
X_valid = (X_valid - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

X_train = X_train[..., np.newaxis]
X_valid = X_valid[..., np.newaxis]
X_test = X_test[..., np.newaxis]

In [ ]:
print(X_train.shape)

In [ ]:
print(X_valid.shape)
print(X_test.shape)

In [ ]:
model=keras.models.Sequential([
    keras.layers.Conv2D(64, kernel_size=(7,7), activation='relu', padding='SAME',input_shape=[28,28,1]),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(128, kernel_size=(5,5), activation='relu', padding='SAME'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(256, kernel_size=(3,3), activation='relu', padding='SAME'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')    
])

Alternatively, for long stacks of layers, we can create a default

In [ ]:
#create a default layer to avoid repeating
from functools import partial
DefaultConv2D = partial(keras.layers.Conv2D, kernel_size=(3,3), activation='relu',padding='SAME')

In [ ]:
model=keras.models.Sequential([
    DefaultConv2D(64, kernel_size=(7,7), input_shape=[28,28,1]),
    keras.layers.MaxPooling2D(),
    DefaultConv2D(128, kernel_size=(5,5)),
    keras.layers.MaxPooling2D(),
    DefaultConv2D(256),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')    
])

Let us try smaller models

In [ ]:
model=keras.models.Sequential([
    DefaultConv2D(64, kernel_size=(5,5), input_shape=[28,28,1]),
    keras.layers.MaxPooling2D(),
    DefaultConv2D(128, kernel_size=(5,5)),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')    
])

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='SGD',metrics=["accuracy"])

In [ ]:
model.summary()

In [ ]:
history=model.fit(X_train,y_train,epochs=10,validation_data=(X_valid,y_valid))

In [ ]:
model.evaluate(X_test,y_test)

In [ ]:
history.history.keys()

In [ ]:
plt.figure(figsize=(8,3))
plt.subplot(121),plt.plot(history.history['accuracy'],label='training');plt.plot(history.history['val_accuracy'],label='validation');plt.title('accuracy')
plt.legend()
plt.subplot(122),plt.plot(history.history['loss'],label='training');plt.plot(history.history['val_loss'],label='validation');plt.title('loss')
plt.legend()

## Compare with 1D network
Now let us compare with the dense network we have been doing before

In [ ]:
model1d = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, kernel_initializer="he_normal",activation="relu"),
    keras.layers.Dense(100, kernel_initializer="he_normal",activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])
model1d.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(lr=1e-3),
              metrics=["accuracy"])

In [ ]:
model1d.summary()

In [ ]:
print(X_train[:,:,:,0].shape)
print(y_train.shape)

In [ ]:


history1d=model1d.fit(X_train[:,:,:,0],y_train,epochs=10,validation_data=(X_valid[:,:,:,0],y_valid))

In [ ]:
model1d.evaluate(X_test[:,:,:,0],y_test)

In [ ]:
plt.figure(figsize=(8,3))
plt.subplot(121),plt.plot(history1d.history['accuracy'],label='training');plt.plot(history1d.history['val_accuracy'],label='validation');plt.title('accuracy')
plt.legend()
plt.subplot(122),plt.plot(history1d.history['loss'],label='training');plt.plot(history1d.history['val_loss'],label='validation');plt.title('loss')
plt.legend();

In [ ]:
plt.figure(figsize=(8,3))
plt.subplot(121),plt.plot(history.history['val_accuracy'],label='conv2d');plt.plot(history1d.history['val_accuracy'],label='dense');plt.title('val_accuracy')
plt.legend()
plt.subplot(122),plt.plot(history.history['val_loss'],label='conv2d');plt.plot(history1d.history['val_loss'],label='dense');plt.title('val_loss')
plt.legend()

In [ ]:
ntest=20;
y_predict2d=model.predict(X_test[:ntest])

In [ ]:
y_predict1d=model1d.predict(X_test[:ntest,:,:,0])

In [ ]:
print(y_test[:ntest])

In [ ]:
print(y_predict2d.shape)

In [ ]:
print(y_predict1d.shape)

In [ ]:
print(['index','true','pred2d','pred1d'])
[print(i,'\t',y_test[i], '\t',y_predict2d[i,:].argmax(), '\t',y_predict1d[i,:].argmax()) for i in range(0,ntest)];

In [ ]:
[plt.imshow(X_test[8,:,:,0]) for i in range(0,10)];

## Exercise 3: Fashion data set
This is similar to MNIST in size but harder to classify right.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train, X_valid = X_train_full[:-5000], X_train_full[-5000:]
y_train, y_valid = y_train_full[:-5000], y_train_full[-5000:]

X_mean = X_train.mean(axis=0, keepdims=True)
X_std = X_train.std(axis=0, keepdims=True) + 1e-7
X_train = (X_train - X_mean) / X_std
X_valid = (X_valid - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

X_train = X_train[..., np.newaxis]
X_valid = X_valid[..., np.newaxis]
X_test = X_test[..., np.newaxis]
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
from functools import partial

DefaultConv2D = partial(keras.layers.Conv2D,
                        kernel_size=3, activation='relu', padding="SAME")

model = keras.models.Sequential([
    DefaultConv2D(filters=64, kernel_size=7, input_shape=[28, 28, 1]),
    keras.layers.MaxPooling2D(pool_size=2),
    DefaultConv2D(filters=128),
    DefaultConv2D(filters=128),
    keras.layers.MaxPooling2D(pool_size=2),
    DefaultConv2D(filters=256),
    DefaultConv2D(filters=256),
    keras.layers.MaxPooling2D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(units=128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=10, activation='softmax'),
])

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

## For each layer, the size of the filters can be calculated as

$\sqrt((param-nfeatures)/nfeatures)$

Subtract nfeatures for the biases (they count as one parameter per feature) and filters are 2D $(nfilter^2)$

Example, below, 
conv2d_3  we have
sqrt((3200-64)/64)=7
conv2d_7, we have
sqrt((590080-256)/256)=48



In [ ]:
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=10, validation_data=[X_valid, y_valid])



In [ ]:
score = model.evaluate(X_test, y_test)

In [ ]:
X_new = X_test[:10] # pretend we have new images
y_pred = model.predict(X_new)

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
plt.figure(figsize=(20,20))
for i in range(9):    
    plt.subplot(191+i)
    plt.imshow(X_new[i].reshape(28,28));
    maxp=np.argmax(y_pred[i])
    plt.title('Prob %.2f'%y_pred[i][maxp])
    plt.xlabel(class_names[maxp])